# RF-DETR → TensorRT Export & Inference

Export an RF-DETR detector to a **TensorRT engine** (`.trt`) and run inference on it directly.

TensorRT delivers the lowest latency on NVIDIA GPUs. `export(format="trt")` compiles a GPU-specific FP16
engine in-process via the TensorRT Python API.

| Step | What happens |
|------|--------------|
| **Export** | `RFDETRSmall` → `rfdetr-small.trt` (FP16) |
| **Load** | Deserialize the engine with `TRTInference` (sync mode, no `pycuda`) |
| **Infer** | Preprocess → run engine → post-process → `supervision` detections |

> **Portability**: a `.trt` engine is locked to the **GPU architecture and TensorRT version** of the
> machine that built it. Build it on the same GPU family you deploy on. For a portable, multi-backend
> path that manages its own engine, prefer `inference-models` (see the closing note).

## 1. Install

The `[tensorrt]` extra pulls in `tensorrt` + `polygraphy` (no `trtexec` binary needed). Requires a
CUDA GPU with a matching NVIDIA driver.

`torchaudio` is uninstalled: RF-DETR never uses it, but `transformers` imports it if present, and a
`torch`/`torchaudio` CUDA-version mismatch (common on Colab) makes that import crash `import rfdetr`.

> **Colab**: select a GPU runtime (**Runtime → Change runtime type → GPU**) before running.

In [ ]:
!pip install -q "rfdetr[onnx,tensorrt]>=1.9.0" supervision pillow
!pip uninstall -q -y torchaudio

## 2. Setup and GPU check

TensorRT export and inference both require CUDA — fail fast with a clear message if no GPU is visible.

In [ ]:
from pathlib import Path

import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError("TensorRT export and inference require a CUDA GPU; none is available.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

EXPORT_DIR = Path("export_tensorrt")
EXPORT_DIR.mkdir(exist_ok=True)
CONFIDENCE_THRESHOLD = 0.5

## 3. Sample image

A single street scene with several COCO classes (dog, bicycle, car) is enough to verify the engine
produces correct detections. The image is downloaded once and reused.

In [ ]:
import urllib.request

from PIL import Image

IMAGE_URL = "https://media.roboflow.com/notebooks/examples/dog.jpeg"
IMAGE_PATH = EXPORT_DIR / "sample.jpg"
if not IMAGE_PATH.exists():
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Sample image: {image.size[0]}×{image.size[1]}")

## 4. Export to a TensorRT engine

`format="trt"` (alias of `"tensorrt"`) compiles the engine in one call. The COCO-pretrained `RFDETRSmall`
is used directly — pass `pretrain_weights="<path/to/checkpoint.pth>"` to export your own fine-tuned model
instead. Engine compilation is the slow step (tens of seconds to a few minutes); it runs once and writes
the `.trt` file to `output_dir`.

The engine is FP16 by default (lowest latency); it automatically falls back to FP32 (with a warning) if
your TensorRT build doesn't expose the FP16 builder flag. Pass `fp16=False` to force FP32 explicitly.

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

engine_path = model.export(format="trt", output_dir=str(EXPORT_DIR))
print(f"TensorRT engine: {engine_path}  ({engine_path.stat().st_size / 1e6:.1f} MB)")

## 5. Load the engine

`TRTInference` deserializes the `.trt` file and allocates GPU I/O bindings. `sync_mode=True` uses
`execute_v2` — a blocking call that needs no `pycuda` (async mode requires a CUDA stream and `pycuda`).
The engine's input tensor shape drives preprocessing so the image is resized to exactly what the graph
expects.

In [ ]:
from rfdetr.export.benchmark import TRTInference, infer_transforms, post_process

trt_model = TRTInference(str(engine_path), device="cuda:0", sync_mode=True)

input_name = trt_model.input_names[0]
_, _, input_h, input_w = trt_model.bindings[input_name].shape
print(f"Engine input '{input_name}': {input_h}×{input_w}")

transforms = infer_transforms((input_h, input_w))

## 6. Run inference

The pipeline mirrors what `predict()` does internally, but against the raw engine:

1. **Preprocess** — resize to the engine resolution, ImageNet-normalize, add a batch dim.
2. **Execute** — the engine returns `dets` (boxes, `cxcywh`, normalized) and `labels` (class logits).
3. **Post-process** — `post_process` runs sigmoid + top-k, converts to `xyxy`, and rescales boxes to the
   original image size.

In [ ]:
image_tensor, _ = transforms(image, None)
blob = {input_name: image_tensor[None].to("cuda:0")}

outputs = trt_model(blob)
trt_model.synchronize()

orig_h, orig_w = image.height, image.width
target_sizes = torch.tensor([[orig_h, orig_w]], device="cuda:0")
result = post_process(outputs, target_sizes)[0]
print(f"Raw detections returned by the engine: {len(result['scores'])}")

## 7. Visualize with supervision

Wrap the post-processed boxes in a `supervision.Detections`, filter by confidence, and annotate the
original image with class names from the COCO label set.

In [ ]:
import supervision as sv

from rfdetr.assets.coco_classes import COCO_CLASSES

detections = sv.Detections(
    xyxy=result["boxes"].cpu().numpy(),
    confidence=result["scores"].cpu().numpy(),
    class_id=result["labels"].cpu().numpy().astype(int),
)
detections = detections[detections.confidence > CONFIDENCE_THRESHOLD]

# COCO-pretrained models emit sparse COCO category IDs (1–90) as class_id — look names up in the
# COCO_CLASSES {id: name} dict, not a 0-based list. (Fine-tuned models use 0-based class_names.)
labels = [
    f"{COCO_CLASSES.get(int(c), str(int(c)))} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)
]
print(f"Kept {len(detections)} detections above {CONFIDENCE_THRESHOLD}: {labels}")

annotated = sv.BoxAnnotator(thickness=3).annotate(scene=np.array(image).copy(), detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.6, text_thickness=1, text_padding=4).annotate(
    scene=annotated, detections=detections, labels=labels
)

OUTPUT_PATH = EXPORT_DIR / "annotated_tensorrt.jpg"
Image.fromarray(annotated).save(OUTPUT_PATH)
print(f"Saved annotated image: {OUTPUT_PATH}")
sv.plot_image(annotated)

## Next steps

- **Production inference** — [`inference-models`](https://github.com/roboflow/inference/tree/main/inference_models)
  is the recommended way to serve RF-DETR. It builds and manages its own TensorRT engine and exposes a
  unified API across PyTorch / ONNX / TensorRT backends, so you do **not** feed it the `.trt` produced here:

  ```python
  from inference_models import AutoModel, BackendType

  model = AutoModel.from_pretrained("rfdetr-small", backend=BackendType.TRT)
  detections = model(image)[0].to_supervision()
  ```

- **Custom resolution** — pass `shape=(H, W)` to `export()` (each dim divisible by the model's block size).
- **Fine-tuned weights** — pass `pretrain_weights="<path/to/checkpoint.pth>"` when constructing the model.
- See the [Export documentation](https://rfdetr.roboflow.com/learn/export/) for all formats and options.